# Phase 5 (LSI) - data prep
Pull gated HF data, clean, audit, analyse, tokenise with the Phase-1 tokenizer, build folds.

**Before running:** Internet ON, Kaggle secret `HF_TOKEN` (or `HF`), attach the Phase-1 tokenizer dataset.

**Output:** `/kaggle/working/lsi_data_clean/`

## Decisions and reasons

| Topic | Decision | Why |
|---|---|---|
| Test IDs in train/dev | Drop those train/dev docs (`EXCLUDE_TEST_OVERLAP_FROM_POOL`) | Avoids training on documents whose ID is also in test. Same rule as Phase 4. |
| Same text, different labels | Drop every copy | The label is unknown, so no copy is trusted. |
| Exact duplicate text | Keep one copy | Duplicates would over-weight a document in training. |
| Test text found in train/dev | Reported, never dropped, never used as labels unless validated in section 7b | Test must keep one prediction per doc. Copying labels is only allowed if dev shows it helps. |
| Rare statutes | Bottom 10% of train counts (`RARE_QUANTILE`) | An absolute cutoff of 5 flagged nothing on this data. |
| Chunks | `MAX_CHUNKS = 32` | Covers about 99.8% of docs; 16 covered about 99.2%. |
| PAD vs SEP | PAD = a new id (`VOCAB_SIZE`), separate from SEP and unk; masks still come from lengths | The Phase-1 tokenizer uses id 0 for SEP, PAD and unk, so `ids != pad` would hide the end-of-chunk token. |

## 1. Configuration

In [ ]:
import os
import re
import sys
import json
import glob
import time
import pickle
import shutil
import hashlib
import unicodedata
from collections import Counter

import numpy as np
import pandas as pd

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
SPLITS = ["train", "dev", "test"]

# Env vars are only for local testing. On Kaggle leave them alone.
LOCAL_DATA_DIR = os.environ.get("LSI_LOCAL_DATA_DIR", None)
TOKENIZER_DIR = os.environ.get("LSI_TOKENIZER_DIR", None)
OUT_DIR = os.environ.get("LSI_OUT_DIR", "/kaggle/working/lsi_data_clean")

SEED = 42
N_FOLDS = 5

# Cleaning
KEEP_NEWLINES = True        # keep paragraph breaks (useful for section analysis)
EMPTY_PLACEHOLDER = "."     # used for an empty test text (a test case is never dropped)
SHORT_DOC_CHARS = 200       # only a warning

# Leakage handling (same decision as Phase 4)
EXCLUDE_TEST_OVERLAP_FROM_POOL = True   # drop train/dev docs whose id also appears in test

# Statutes
RARE_QUANTILE = 0.10        # statutes in the bottom 10% of train counts are rare
RARE_THRESHOLD = None        # set in section 7 from RARE_QUANTILE

# Long-document views
CHUNK_LEN = 1024            # tokens per chunk, incl. the final SEP
CHUNK_OVERLAP = 128
MAX_CHUNKS = 32
HEAD_TAIL_LEN = 1024
HEAD_FRAC = 0.5

# Quick text baselines (CPU) to see which part of a document carries signal
BASELINE_MAX_TRAIN = 5000
BASELINE_MAX_DEV = 3000
BASELINE_MIN_POS = 5        # only statutes with at least this many train positives

os.makedirs(OUT_DIR, exist_ok=True)
np.random.seed(SEED)
print("OUT_DIR:", OUT_DIR)

## 2. Hugging Face login

In [ ]:
HF_TOKEN = None

if LOCAL_DATA_DIR is None:
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        for secret_name in ["HF_TOKEN", "HF"]:
            if HF_TOKEN is None:
                try:
                    HF_TOKEN = secrets.get_secret(secret_name)
                    print("Using Kaggle secret:", secret_name)
                except Exception:
                    pass
    except ImportError:
        print("kaggle_secrets not available (not on Kaggle).")

    if HF_TOKEN is None:
        HF_TOKEN = os.environ.get("HF_TOKEN", None)

    if HF_TOKEN is None:
        raise RuntimeError("No HF token found. Add a Kaggle secret named HF_TOKEN (or HF).")

    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("Hugging Face login successful.")
else:
    print("Local data mode, skipping Hugging Face login.")

## 3. Find and download the LSI files (train / dev / test)

In [ ]:
RAW_DIR = os.path.join(OUT_DIR, "raw_snapshot")
os.makedirs(RAW_DIR, exist_ok=True)


def simple_name(path):
    name = path.lower()
    name = name.replace("_", "")
    name = name.replace("-", "")
    name = name.replace(" ", "")
    return name


def is_lsi_path(path):
    key = simple_name(path)
    return ("lsi" in key) or ("phase5" in key)


candidates = {"train": [], "dev": [], "test": []}
all_files = []

if LOCAL_DATA_DIR is not None:
    for root, dirs, files in os.walk(LOCAL_DATA_DIR):
        for file_name in files:
            full_path = os.path.join(root, file_name)
            all_files.append(full_path)
            stem, ext = os.path.splitext(file_name.lower())
            if ext not in [".json", ".jsonl"]:
                continue
            if stem in candidates and is_lsi_path(full_path):
                candidates[stem].append(full_path)
else:
    from huggingface_hub import list_repo_files
    all_files = list_repo_files(HF_DATASET_REPO, repo_type="dataset", token=HF_TOKEN)
    print("Files in repo:", len(all_files))
    for path in all_files:
        if not is_lsi_path(path):
            continue
        stem, ext = os.path.splitext(os.path.basename(path).lower())
        if ext not in [".json", ".jsonl"]:
            continue
        if stem in candidates:
            candidates[stem].append(path)

for split in SPLITS:
    print(split, "->", candidates[split])

for split in SPLITS:
    if len(candidates[split]) == 0:
        print("\nAll repo files, to help you spot the right ones:")
        for p in all_files:
            print("  ", p)
        raise FileNotFoundError("No file found for split: " + split)

SPLIT_PATH = {}
for split in SPLITS:
    options = sorted(candidates[split], key=len)
    chosen = options[0]
    if len(options) > 1:
        print("WARNING: several candidates for", split, "- using the shortest path:", chosen)
    if LOCAL_DATA_DIR is not None:
        local_path = chosen
    else:
        from huggingface_hub import hf_hub_download
        local_path = hf_hub_download(HF_DATASET_REPO, chosen, repo_type="dataset", token=HF_TOKEN)
    ext = os.path.splitext(local_path)[1]
    saved_copy = os.path.join(RAW_DIR, split + ext)
    shutil.copy(local_path, saved_copy)
    SPLIT_PATH[split] = saved_copy
    print(split, "ready:", saved_copy, "|", round(os.path.getsize(saved_copy) / 1e6, 1), "MB")

## 4. Load (tolerant to several JSON layouts)

LSI labels are a **list of statute IDs** per document (multi-label).

In [ ]:
ID_KEYS = ["id", "case_id", "caseid", "doc_id", "ID", "Id"]
TEXT_KEYS = ["text", "document", "case_text", "judgment_text", "content", "Text"]
LABEL_KEYS = ["labels", "label", "statutes", "statute_ids", "statute", "y", "Labels"]
LIST_KEYS = ["data", "examples", "records", "cases", "documents", "items"]


def read_any(path):
    with open(path, "r", encoding="utf-8-sig") as f:
        raw = f.read().strip()
    if raw == "":
        return []
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        rows = []
        for line in raw.splitlines():
            line = line.strip()
            if line == "":
                continue
            rows.append(json.loads(line))
        return rows


def first_present(record, keys):
    for k in keys:
        if k in record:
            return k
    return None


def to_record_list(obj):
    # Returns a list of (key, record_dict_or_text)
    out = []
    if isinstance(obj, dict):
        found_list = None
        for k in LIST_KEYS:
            if k in obj and isinstance(obj[k], list):
                found_list = obj[k]
        if found_list is not None:
            obj = found_list
        else:
            for key, value in obj.items():
                out.append((key, value))
            return out
    for i in range(len(obj)):
        out.append((None, obj[i]))
    return out


ANOMALY = Counter()          # every anomaly counted here, reported in section 6b
ANOMALY_VALUES = Counter()   # the raw non-positive label values, so you can see what they are
CURRENT_SPLIT = "train"


def label_to_statute_list(value):
    # Returns a sorted list of unique positive int statute IDs. Unusable values give [].
    # Every problem is counted in ANOMALY, nothing is dropped silently.
    if value is None:
        ANOMALY["label_missing_" + CURRENT_SPLIT] += 1
        return []
    if isinstance(value, str):
        text_value = value.strip()
        try:
            value = json.loads(text_value)
            ANOMALY["label_stored_as_string"] += 1
        except json.JSONDecodeError:
            value = [x for x in re.split(r"[,\s;]+", text_value) if x != ""]
            ANOMALY["label_stored_as_plain_text"] += 1
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        value = [value]
    if not isinstance(value, list):
        ANOMALY["label_wrong_type"] += 1
        return []
    if len(value) == 0:
        ANOMALY["label_empty_list_" + CURRENT_SPLIT] += 1
        return []
    good = []
    for item in value:
        try:
            num = int(float(item))
        except (TypeError, ValueError):
            ANOMALY["label_unparseable_item"] += 1
            continue
        if num > 0:
            good.append(num)
        else:
            ANOMALY["label_nonpositive_item"] += 1
            ANOMALY_VALUES[repr(item)] += 1
    if len(good) != len(set(good)):
        ANOMALY["label_repeated_in_doc"] += 1
    return sorted(set(good))


raw_docs = {}
for split in SPLITS:
    CURRENT_SPLIT = split
    obj = read_any(SPLIT_PATH[split])
    pairs = to_record_list(obj)
    docs = []
    bad_label_count = 0
    for pos in range(len(pairs)):
        key, rec = pairs[pos]
        if isinstance(rec, str):
            doc_id = key
            text = rec
            label_raw = None
        else:
            id_key = first_present(rec, ID_KEYS)
            text_key = first_present(rec, TEXT_KEYS)
            label_key = first_present(rec, LABEL_KEYS)
            if id_key is not None:
                doc_id = rec[id_key]
            elif key is not None:
                doc_id = key
            else:
                doc_id = "doc_" + str(pos)
            if text_key is None:
                raise KeyError("No text field in record. Keys seen: " + str(list(rec.keys())))
            text = rec[text_key]
            label_raw = rec[label_key] if label_key is not None else None
        labels = label_to_statute_list(label_raw)
        if split != "test" and len(labels) == 0:
            bad_label_count += 1
        if split == "test":
            labels = []
        docs.append({"id": doc_id, "text": text, "labels": labels})
    raw_docs[split] = docs
    print(split, "| docs:", len(docs), "| docs without usable labels:", bad_label_count)

if len(raw_docs["test"]) == 0:
    print("WARNING: test file is empty right now. The spec says data is dynamic - re-run later.")
if len(raw_docs["train"]) == 0:
    raise RuntimeError("train file has 0 docs.")
print("\nFirst train record check:")
first_train = raw_docs["train"][0]
print({"id": first_train["id"], "labels": first_train["labels"], "text[:150]": str(first_train["text"])[:150]})

## 5. Clean

In [ ]:
# Build the character map from code numbers only (keeps this cell pure ASCII).
CHAR_MAP = {}
for cp in range(0, 9):
    CHAR_MAP[cp] = None            # control chars: delete
CHAR_MAP[9] = " "                  # tab
CHAR_MAP[11] = " "                 # vertical tab
CHAR_MAP[12] = " "                 # form feed
for cp in range(14, 32):
    CHAR_MAP[cp] = None
for cp in range(127, 160):
    CHAR_MAP[cp] = None
for cp in [0x200B, 0x200C, 0x200D, 0x2060, 0xFEFF, 0x00AD, 0xFFFD]:
    CHAR_MAP[cp] = None            # zero-width chars, soft hyphen, replacement char: delete
SPACE_CODES = [0x00A0, 0x1680, 0x202F, 0x205F, 0x3000]
for cp in range(0x2000, 0x200B):
    SPACE_CODES.append(cp)
for cp in SPACE_CODES:
    CHAR_MAP[cp] = " "             # odd spaces -> normal space
CHAR_MAP[0x2028] = "\n"            # line / paragraph separators -> newline
CHAR_MAP[0x2029] = "\n"

MANY_SPACES = re.compile(r" {2,}")
MANY_NEWLINES = re.compile(r"\n{3,}")


def clean_text(text):
    if text is None:
        text = ""
    if not isinstance(text, str):
        text = str(text)
    text = unicodedata.normalize("NFC", text)
    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")
    text = text.translate(CHAR_MAP)
    lines = text.split("\n")
    cleaned_lines = []
    for line in lines:
        line = MANY_SPACES.sub(" ", line)
        line = line.strip()
        cleaned_lines.append(line)
    text = "\n".join(cleaned_lines)
    if KEEP_NEWLINES:
        text = MANY_NEWLINES.sub("\n\n", text)
    else:
        text = text.replace("\n", " ")
        text = MANY_SPACES.sub(" ", text)
    text = text.strip()
    return text


clean_docs = {}
drop_log = []
clean_stats = {"changed_docs": 0, "chars_removed": 0}

for split in SPLITS:
    kept = []
    seen_ids = set()
    for d in raw_docs[split]:
        original_text = d["text"] if isinstance(d["text"], str) else str(d["text"])
        text = clean_text(original_text)
        if text != original_text:
            clean_stats["changed_docs"] += 1
            clean_stats["chars_removed"] += max(0, len(original_text) - len(text))
        id_str = str(d["id"])
        if split != "test":
            if len(d["labels"]) == 0:
                drop_log.append((split, d["id"], "no usable labels"))
                continue
            if text == "":
                drop_log.append((split, d["id"], "empty text"))
                continue
            if id_str in seen_ids:
                drop_log.append((split, d["id"], "duplicate id"))
                continue
        else:
            if text == "":
                text = EMPTY_PLACEHOLDER
                print("test doc with empty text kept with placeholder:", d["id"])
            if id_str in seen_ids:
                print("WARNING: duplicate id inside test (kept, check with TAs):", d["id"])
        seen_ids.add(id_str)
        kept.append({"id": d["id"], "text": text, "labels": d["labels"]})
    clean_docs[split] = kept
    print(split, "raw:", len(raw_docs[split]), "-> kept:", len(kept))

print("\nCleaning touched", clean_stats["changed_docs"], "docs, removed", clean_stats["chars_removed"], "chars")
print("Dropped so far:", len(drop_log))
for item in drop_log[:15]:
    print("  ", item)

## 6. Audit: duplicates, conflicts, leakage

In [ ]:
def text_hash(text):
    return hashlib.sha1(text.encode("utf-8")).hexdigest()


def head_hash(text, n=500):
    return hashlib.sha1(text[:n].lower().encode("utf-8")).hexdigest()


# keep a copy of dev before de-duplication, so section 7b can still see dev docs that repeat train text
dev_before_dedup = [dict(d) for d in clean_docs["dev"]]

# 6a. exact duplicate texts inside train+dev (train first, so train copy wins)
seen_hash = {}
conflict_hashes = set()
for split in ["train", "dev"]:
    for d in clean_docs[split]:
        h = text_hash(d["text"])
        labels_key = tuple(d["labels"])
        if h in seen_hash:
            if seen_hash[h] != labels_key:
                conflict_hashes.add(h)
        else:
            seen_hash[h] = labels_key

dup_removed = 0
conflict_removed = 0
emitted = set()
for split in ["train", "dev"]:
    kept = []
    for d in clean_docs[split]:
        h = text_hash(d["text"])
        if h in conflict_hashes:
            drop_log.append((split, d["id"], "same text with conflicting statute labels"))
            conflict_removed += 1
            continue
        if h in emitted:
            drop_log.append((split, d["id"], "exact duplicate text"))
            dup_removed += 1
            continue
        emitted.add(h)
        kept.append(d)
    clean_docs[split] = kept
print("exact duplicates removed:", dup_removed, "| conflicting-label copies removed:", conflict_removed)

# 6b. id overlap across splits (report)
train_ids = set(str(d["id"]) for d in clean_docs["train"])
dev_ids = set(str(d["id"]) for d in clean_docs["dev"])
test_ids = set(str(d["id"]) for d in clean_docs["test"])
print("id overlap train/dev:", len(train_ids & dev_ids), "| train/test:", len(train_ids & test_ids), "| dev/test:", len(dev_ids & test_ids))

# 6c. test vs train/dev text overlap (report)
labelled_hashes = set()
labelled_head_hashes = set()
for split in ["train", "dev"]:
    for d in clean_docs[split]:
        labelled_hashes.add(text_hash(d["text"]))
        labelled_head_hashes.add(head_hash(d["text"]))

test_exact = 0
test_head = 0
for d in clean_docs["test"]:
    if text_hash(d["text"]) in labelled_hashes:
        test_exact += 1
    if head_hash(d["text"]) in labelled_head_hashes:
        test_head += 1
print("test docs whose exact text is also in train/dev:", test_exact, "of", len(clean_docs["test"]))
print("test docs whose first 500 chars match a train/dev doc:", test_head)

# 6d. drop train/dev docs whose id is in test (same decision as Phase 4)
n_removed_overlap = 0
if EXCLUDE_TEST_OVERLAP_FROM_POOL:
    for split in ["train", "dev"]:
        kept = []
        for d in clean_docs[split]:
            if str(d["id"]) in test_ids:
                drop_log.append((split, d["id"], "id also in test (leakage)"))
                n_removed_overlap += 1
                continue
            kept.append(d)
        clean_docs[split] = kept
print("train/dev docs removed because their id is also in test:", n_removed_overlap)

# 6e. very short docs
short_counts = {}
for split in SPLITS:
    n_short = 0
    for d in clean_docs[split]:
        if len(d["text"]) < SHORT_DOC_CHARS:
            n_short += 1
    short_counts[split] = n_short
print("docs shorter than", SHORT_DOC_CHARS, "chars:", short_counts)

for split in SPLITS:
    print(split, "final docs:", len(clean_docs[split]))

audit = {
    "raw_counts": {s: len(raw_docs[s]) for s in SPLITS},
    "final_counts": {s: len(clean_docs[s]) for s in SPLITS},
    "dropped": [list(map(str, x)) for x in drop_log],
    "exact_dup_removed": dup_removed,
    "conflict_removed": conflict_removed,
    "removed_test_overlap_from_pool": n_removed_overlap,
    "test_exact_in_labelled": test_exact,
    "test_head_match_labelled": test_head,
    "short_docs": short_counts,
    "clean_stats": clean_stats,
}

## 6b. Anomaly audit and handling

Every anomaly is counted, flagged per document, and handled by the rule in the table.

In [ ]:
TAG_RE = re.compile(r"<[A-Z][A-Z_]{1,20}>")

print("=== Load-time label anomalies ===")
if len(ANOMALY) == 0:
    print("none")
for key in sorted(ANOMALY.keys()):
    print("  ", key, ANOMALY[key])

anomaly_rows = []
junk_ids = {"train": set(), "dev": set()}
tag_counter = Counter()
for split in SPLITS:
    for d in clean_docs[split]:
        text = d["text"]
        tags = TAG_RE.findall(text)
        tag_counter.update(tags)
        stripped = TAG_RE.sub(" ", text)
        no_letters = not any(ch.isalpha() for ch in stripped)
        is_short = len(text) < SHORT_DOC_CHARS
        n_labels = len(d["labels"])
        flags = []
        if no_letters:
            flags.append("no_letters_or_tags_only")
        if is_short:
            flags.append("very_short")
        if len(tags) > 0:
            flags.append("has_placeholder_tags")
        if n_labels >= 20:
            flags.append("many_labels")
        if len(text.split()) > 20000:
            flags.append("very_long")
        if split != "test" and no_letters:
            junk_ids[split].add(str(d["id"]))
        anomaly_rows.append({
            "split": split,
            "id": str(d["id"]),
            "n_chars": len(text),
            "n_words": len(text.split()),
            "n_labels": n_labels,
            "n_tags": len(tags),
            "flags": ";".join(flags),
        })

anomaly_df = pd.DataFrame(anomaly_rows)
print("\n=== Raw non-positive label values (top 20) ===")
print(ANOMALY_VALUES.most_common(20))
zero_keys = ["0", "0.0", "'0'", "'0.0'"]
if any(k in ANOMALY_VALUES for k in zero_keys):
    print("WARNING: label value 0 found. Check whether 0 means 'no statute' before trusting the labels.")
print("Rule used: non-positive values are dropped; a doc left with no labels is dropped from train/dev.")

print("\n=== Document flags by split ===")
for flag in ["no_letters_or_tags_only", "very_short", "has_placeholder_tags", "many_labels", "very_long"]:
    counts = {}
    for split in SPLITS:
        sub = anomaly_df[anomaly_df["split"] == split]
        counts[split] = int(sub["flags"].str.contains(flag).sum())
    print(flag.ljust(24), counts)

print("\n=== Most common placeholder tags (all splits) ===")
print(tag_counter.most_common(15))

# Handling: docs with no words at all (only tags or punctuation) cannot teach the model.
# They are dropped from train/dev. They stay in test, because every test doc needs a prediction.
for split in ["train", "dev"]:
    before = len(clean_docs[split])
    kept = []
    for d in clean_docs[split]:
        if str(d["id"]) in junk_ids[split]:
            drop_log.append((split, d["id"], "no letters or tags only (junk)"))
            continue
        kept.append(d)
    clean_docs[split] = kept
    print(split, "junk docs removed:", before - len(kept))

# Short docs are kept. Their OOF scores are reported separately in the training notebook.
print("short docs kept (flag very_short):", int(anomaly_df["flags"].str.contains("very_short").sum()))

anomaly_df.to_csv(os.path.join(OUT_DIR, "anomaly_report.csv"), index=False)
audit["anomalies_load_time"] = dict(ANOMALY)
audit["anomaly_flag_counts"] = {
    flag: int(anomaly_df["flags"].str.contains(flag).sum())
    for flag in ["no_letters_or_tags_only", "very_short", "has_placeholder_tags", "many_labels", "very_long"]
}
audit["junk_removed"] = {split: len(junk_ids[split]) for split in ["train", "dev"]}
print("\nfinal docs after anomaly handling:", {s: len(clean_docs[s]) for s in SPLITS})

## 7. Label analysis: statutes, rare ones, co-occurrence

In [ ]:
from itertools import combinations

train_statute_counts = Counter()
for d in clean_docs["train"]:
    train_statute_counts.update(d["labels"])

dev_statute_counts = Counter()
for d in clean_docs["dev"]:
    dev_statute_counts.update(d["labels"])

pool_statute_counts = train_statute_counts + dev_statute_counts

print("=== Statute coverage ===")
print("distinct statutes in train:", len(train_statute_counts))
print("distinct statutes in dev:", len(dev_statute_counts))
print("distinct statutes in train+dev:", len(pool_statute_counts))

unseen_dev = [s for s in dev_statute_counts if s not in train_statute_counts]
print("dev statutes never seen in train:", len(unseen_dev), sorted(unseen_dev)[:30])

RARE_THRESHOLD = int(np.percentile(list(train_statute_counts.values()), 100 * RARE_QUANTILE))
rare_in_train = [s for s, c in train_statute_counts.items() if c <= RARE_THRESHOLD]
print("rare threshold (bottom", int(100 * RARE_QUANTILE), "% of train counts): count <=", RARE_THRESHOLD)
print("rare statutes in train:", len(rare_in_train))

print("\nMost common statutes (train):", train_statute_counts.most_common(10))
print("Least common statutes (train):", sorted(train_statute_counts.items(), key=lambda x: x[1])[:10])

print("\n=== Labels per document ===")
for split in ["train", "dev"]:
    counts_here = [len(d["labels"]) for d in clean_docs[split]]
    if len(counts_here) == 0:
        continue
    print(split, "| min", min(counts_here), "| mean %.2f" % (sum(counts_here) / len(counts_here)),
          "| max", max(counts_here), "| single-label docs:", sum(1 for n in counts_here if n == 1))

print("\n=== Top co-occurring statute pairs (train) ===")
pair_counts = Counter()
for d in clean_docs["train"]:
    for a, b in combinations(d["labels"], 2):
        pair_counts[(a, b)] += 1
print(pair_counts.most_common(15))

# label table for later (train + dev)
label_rows = []
for s in sorted(pool_statute_counts.keys()):
    label_rows.append({
        "statute_id": s,
        "train_count": train_statute_counts.get(s, 0),
        "dev_count": dev_statute_counts.get(s, 0),
        "is_rare": (train_statute_counts.get(s, 0) <= RARE_THRESHOLD),
    })
label_table = pd.DataFrame(label_rows)
print("\nlabel table rows:", len(label_table))
label_table.head(10)

## 7b. Exact-text lookup check (is copying train labels useful on dev?)

Test docs with identical text in train/dev can only be scored by a lookup if dev shows the lookup is accurate.

In [ ]:
train_by_hash = {}
for d in clean_docs["train"]:
    train_by_hash[text_hash(d["text"])] = d["labels"]

dev_hits = []
for d in dev_before_dedup:
    h = text_hash(d["text"])
    if h in train_by_hash:
        dev_hits.append((d["labels"], train_by_hash[h]))

print("dev docs with exact text in train:", len(dev_hits), "of", len(clean_docs["dev"]))
if len(dev_hits) > 0:
    exact_match = sum(1 for a, b in dev_hits if a == b)
    print("exact label-set agreement:", exact_match, "of", len(dev_hits),
          "=", round(100.0 * exact_match / len(dev_hits), 1), "%")
    tp = 0
    fp = 0
    fn = 0
    for true_labels, copied_labels in dev_hits:
        true_set = set(true_labels)
        copy_set = set(copied_labels)
        tp += len(true_set & copy_set)
        fp += len(copy_set - true_set)
        fn += len(true_set - copy_set)
    micro_p = tp / max(1, tp + fp)
    micro_r = tp / max(1, tp + fn)
    micro_f1 = 2 * micro_p * micro_r / max(1e-9, micro_p + micro_r)
    print("micro-F1 of copying train labels on these dev docs:", round(micro_f1, 4))
    print("Use the lookup for test only if this is clearly above the model's dev score on the same docs.")
else:
    print("No exact text matches, lookup not used.")

## 8. Length analysis and plots

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

rows = []
for split in SPLITS:
    for d in clean_docs[split]:
        words = d["text"].split()
        paras = 0
        for p in d["text"].split("\n"):
            if p.strip() != "":
                paras += 1
        rows.append({
            "split": split,
            "id": str(d["id"]),
            "n_labels": len(d["labels"]),
            "n_chars": len(d["text"]),
            "n_words": len(words),
            "n_paras": paras,
        })
df_all = pd.DataFrame(rows)

print("=== Length stats (words) by split ===")
print(df_all.groupby("split")["n_words"].describe(percentiles=[0.5, 0.9, 0.99]).round(0))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

lab_df = df_all[df_all["split"].isin(["train", "dev"])]
axes[0].hist(lab_df["n_words"], bins=50)
axes[0].set_title("Words per document (train+dev)")

freq_sorted = sorted(pool_statute_counts.values(), reverse=True)
axes[1].bar(range(len(freq_sorted)), freq_sorted)
axes[1].set_yscale("log")
axes[1].set_title("Statute frequency (train+dev, log scale)")

axes[2].hist(lab_df["n_labels"], bins=range(1, int(lab_df["n_labels"].max()) + 2))
axes[2].set_title("Statutes per document")

plt.tight_layout()
fig_path = os.path.join(OUT_DIR, "analysis.png")
plt.savefig(fig_path, dpi=120)
plt.show()
print("saved", fig_path)

## 8b. Which part of the document carries signal? (TF-IDF + logistic regression)

Quick CPU baseline on a few slices of each document. Macro-F1 over statutes that have enough positives.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score


def make_segment(words, name):
    n = len(words)
    if n == 0:
        return "."
    if name == "full":
        return " ".join(words)
    if name == "first_512_words":
        return " ".join(words[:512])
    if name == "last_512_words":
        return " ".join(words[-512:])
    if name == "head256_tail256":
        if n <= 512:
            return " ".join(words)
        return " ".join(words[:256] + words[-256:])
    if name.startswith("part_"):
        k = int(name.split("_")[1])
        start = int(n * (k - 1) / 5)
        end = int(n * k / 5)
        piece = words[start:end]
        if len(piece) == 0:
            return "."
        return " ".join(piece)
    raise ValueError(name)


rng = np.random.RandomState(SEED)
train_docs = clean_docs["train"]
dev_docs = clean_docs["dev"]

if len(train_docs) > BASELINE_MAX_TRAIN:
    idx = rng.choice(len(train_docs), BASELINE_MAX_TRAIN, replace=False)
    base_train = [train_docs[int(i)] for i in idx]
else:
    base_train = train_docs

if len(dev_docs) > BASELINE_MAX_DEV:
    idx = rng.choice(len(dev_docs), BASELINE_MAX_DEV, replace=False)
    base_dev = [dev_docs[int(i)] for i in idx]
else:
    base_dev = dev_docs

print("baseline train docs:", len(base_train), "| dev docs:", len(base_dev))

base_train_counts = Counter()
for d in base_train:
    base_train_counts.update(d["labels"])
usable_statutes = sorted([s for s, c in base_train_counts.items() if c >= BASELINE_MIN_POS])
print("statutes used in baseline:", len(usable_statutes))

segment_names = ["full", "first_512_words", "last_512_words", "head256_tail256",
                 "part_1", "part_2", "part_3", "part_4", "part_5"]
baseline_rows = []

if len(base_dev) > 0 and len(usable_statutes) > 1:
    mlb = MultiLabelBinarizer(classes=usable_statutes)
    Y_tr = mlb.fit_transform([[s for s in d["labels"] if s in usable_statutes] for d in base_train])
    Y_dv = mlb.transform([[s for s in d["labels"] if s in usable_statutes] for d in base_dev])

    train_words = [d["text"].split() for d in base_train]
    dev_words = [d["text"].split() for d in base_dev]

    for name in segment_names:
        t0 = time.time()
        tr_texts = [make_segment(w, name) for w in train_words]
        dv_texts = [make_segment(w, name) for w in dev_words]
        vec = TfidfVectorizer(max_features=50000, sublinear_tf=True, min_df=2)
        X_tr = vec.fit_transform(tr_texts)
        X_dv = vec.transform(dv_texts)
        clf = OneVsRestClassifier(LogisticRegression(max_iter=1000, class_weight="balanced"))
        clf.fit(X_tr, Y_tr)
        pred = clf.predict(X_dv)
        macro = f1_score(Y_dv, pred, average="macro", zero_division=0)
        micro = f1_score(Y_dv, pred, average="micro", zero_division=0)
        baseline_rows.append({"slice": name, "macro_F1": round(macro, 4), "micro_F1": round(micro, 4),
                              "sec": round(time.time() - t0, 1)})
        print(name, "| macro-F1:", round(macro, 4), "| micro-F1:", round(micro, 4))

    baseline_df = pd.DataFrame(baseline_rows)
    print()
    print(baseline_df.to_string(index=False))
    print("\npart_1..part_5 = fifths of the document in order (part_5 is the end).")
else:
    baseline_df = pd.DataFrame(baseline_rows)
    print("Skipped baselines (no dev docs or too few usable statutes).")

## 9. Load the Phase-1 tokenizer

In [ ]:
from transformers import AutoTokenizer
os.environ["TOKENIZERS_PARALLELISM"] = "true"


def find_tokenizer_dir():
    if TOKENIZER_DIR is not None:
        return TOKENIZER_DIR
    paths = glob.glob("/kaggle/input/**/tokenizer.json", recursive=True)
    folders = []
    for p in paths:
        folder = os.path.dirname(p)
        if "checkpoint-" not in folder:
            folders.append(folder)
    folders = sorted(set(folders))
    if len(folders) == 0:
        raise FileNotFoundError("No tokenizer.json under /kaggle/input. Attach your tokenizer dataset (Add Input).")
    for folder in folders:
        if os.path.exists(os.path.join(folder, "trained_on.json")):
            return folder
    return folders[0]


tok_dir = find_tokenizer_dir()
tokenizer = AutoTokenizer.from_pretrained(tok_dir)
tokenizer.model_max_length = 10 ** 9
VOCAB_SIZE = len(tokenizer)
print("Tokenizer:", tok_dir, "| vocab size:", VOCAB_SIZE)


def find_sep_id(tok):
    if tok.eos_token_id is not None:
        return int(tok.eos_token_id)
    for name in ["<eos>", "</s>", "<|endoftext|>", "<EOS>"]:
        tid = tok.convert_tokens_to_ids(name)
        if tid is not None and tid != tok.unk_token_id:
            return int(tid)
    raise ValueError("Could not find an EOS-like token to use as separator.")


SEP_ID = find_sep_id(tokenizer)
# PAD gets its own id, one past the last tokenizer id, so it never collides with SEP or unk.
# The Phase-1 embedding table needs one extra row for it (the training notebook adds that).
PAD_ID = VOCAB_SIZE
assert PAD_ID != SEP_ID, "PAD must differ from SEP"
print("SEP (EOS) id:", SEP_ID, "| PAD id (dedicated):", PAD_ID, "| embedding rows needed:", PAD_ID + 1)

if VOCAB_SIZE < 65535:
    ID_DTYPE = np.uint16
else:
    ID_DTYPE = np.int32
print("ids stored as:", ID_DTYPE.__name__)

# Oracle check: for a byte-level BPE, decode(encode(text)) should give the text back.
ok = 0
sample_texts = [d["text"] for d in clean_docs["train"][:30]]
for t in sample_texts:
    ids = tokenizer(t, add_special_tokens=False)["input_ids"]
    back = tokenizer.decode(ids, clean_up_tokenization_spaces=False)
    if back == t:
        ok += 1
print("round-trip exact match:", ok, "/", len(sample_texts))
if ok < len(sample_texts):
    print("Not exact: the tokenizer normalises or lowercases text. Fine as long as it is the same one Phase 1 used.")

## 10. Tokenise each document once (no truncation)

In [ ]:
def tokenize_texts(texts, batch_size=32, name=""):
    all_ids = []
    t0 = time.time()
    n_batches = (len(texts) + batch_size - 1) // batch_size
    for b in range(n_batches):
        batch = texts[b * batch_size:(b + 1) * batch_size]
        out = tokenizer(batch, add_special_tokens=False)
        for ids in out["input_ids"]:
            all_ids.append(np.array(ids, dtype=ID_DTYPE))
        if b % 100 == 0:
            print(name, "batch", b + 1, "/", n_batches, "| elapsed", round(time.time() - t0, 1), "s")
    return all_ids


tokenized = {}
for split in SPLITS:
    texts = [d["text"] for d in clean_docs[split]]
    tokenized[split] = tokenize_texts(texts, name=split)
    print(split, "tokenised:", len(tokenized[split]))

max_id = 0
n_unk = 0
n_tokens_total = 0
unk_id = tokenizer.unk_token_id
for split in SPLITS:
    for arr in tokenized[split]:
        if len(arr) == 0:
            continue
        m = int(arr.max())
        if m > max_id:
            max_id = m
        n_tokens_total += len(arr)
        if unk_id is not None:
            n_unk += int((arr == unk_id).sum())
print("max token id:", max_id, "(vocab", VOCAB_SIZE, ")")
assert max_id < VOCAB_SIZE
print("total tokens:", n_tokens_total, "| unk tokens:", n_unk)

## 11. Token-length analysis and long-document views

In [ ]:
tok_rows = []
for split in SPLITS:
    for i in range(len(clean_docs[split])):
        tok_rows.append({
            "split": split,
            "id": str(clean_docs[split][i]["id"]),
            "n_labels": len(clean_docs[split][i]["labels"]),
            "n_tokens": len(tokenized[split][i]),
        })
df_tok = pd.DataFrame(tok_rows)

print("=== Token length by split ===")
print(df_tok.groupby("split")["n_tokens"].describe(percentiles=[0.5, 0.9, 0.99]).round(0))

print("\n=== % of docs longer than L tokens (needs truncation / chunking) ===")
for L in [512, 1024, 2048, 4096, 8192, 16384]:
    line = "L=" + str(L).rjust(5) + " | "
    for split in SPLITS:
        sub = df_tok[df_tok["split"] == split]["n_tokens"]
        if len(sub) > 0:
            line += split + ": " + str(round(100.0 * (sub > L).mean(), 1)).rjust(5) + "%   "
    print(line)

print("\n=== Chunking summary (chunk_len=%d, overlap=%d) ===" % (CHUNK_LEN, CHUNK_OVERLAP))
for split in ["train", "dev", "test"]:
    counts = []
    for n in df_tok[df_tok["split"] == split]["n_tokens"]:
        body_len = CHUNK_LEN - 1
        stride = body_len - CHUNK_OVERLAP
        starts = []
        s = 0
        while True:
            starts.append(s)
            if s + body_len >= n:
                break
            s += stride
        counts.append(len(starts))
    if len(counts) > 0:
        counts = np.array(counts)
        print(split, "| mean chunks:", round(counts.mean(), 1), "| median:", int(np.median(counts)),
              "| p90:", int(np.percentile(counts, 90)), "| max:", int(counts.max()),
              "| pct over MAX_CHUNKS (" + str(MAX_CHUNKS) + "):", round(100.0 * (counts > MAX_CHUNKS).mean(), 1))

lab_tok = df_tok[df_tok["split"].isin(["train", "dev"])]
if len(lab_tok) > 0:
    print("\nCorrelation of token length with number of statutes:",
          round(lab_tok["n_tokens"].corr(lab_tok["n_labels"]), 3))

## 12. Folds (train+dev combined)

Multi-label stratification: each document is keyed by its **rarest** statute, so rare statutes end up spread across folds.

In [ ]:
from sklearn.model_selection import StratifiedKFold

fold_index = []
fold_keys = []
for split in ["train", "dev"]:
    for i in range(len(clean_docs[split])):
        fold_index.append((split, i))

pool_counts = Counter()
for split in ["train", "dev"]:
    for d in clean_docs[split]:
        pool_counts.update(d["labels"])


def rarest_statute(labels):
    best = labels[0]
    for s in labels:
        if pool_counts[s] < pool_counts[best]:
            best = s
    if pool_counts[best] < N_FOLDS:
        return -1
    return best


for split, i in fold_index:
    fold_keys.append(rarest_statute(clean_docs[split][i]["labels"]))
fold_keys = np.array(fold_keys)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_of_row = np.full(len(fold_keys), -1, dtype=int)
for fold_id, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(fold_keys)), fold_keys)):
    fold_of_row[va_idx] = fold_id

fold_arrays = {
    "train": np.full(len(clean_docs["train"]), -1, dtype=int),
    "dev": np.full(len(clean_docs["dev"]), -1, dtype=int),
    "test": np.full(len(clean_docs["test"]), -1, dtype=int),
}
for row in range(len(fold_index)):
    split, i = fold_index[row]
    fold_arrays[split][i] = fold_of_row[row]

print("fold sizes and rare-statute coverage:")
for f in range(N_FOLDS):
    mask = fold_of_row == f
    fold_statutes = set()
    for row in np.where(mask)[0]:
        split, i = fold_index[row]
        fold_statutes.update(clean_docs[split][i]["labels"])
    print("  fold", f, "| docs:", int(mask.sum()), "| distinct statutes:", len(fold_statutes))

# positive weights for BCE: negatives / positives per statute, from train+dev
n_pool = len(fold_index)
pos_weight = {}
for s, c in pool_counts.items():
    pos_weight[int(s)] = float((n_pool - c) / max(1, c))
print("\npos_weight examples (rarest 5):", sorted(pos_weight.items(), key=lambda x: -x[1])[:5])

## 13. Helpers saved for the training notebooks

In [ ]:
%%writefile lsi_utils.py

import json
import numpy as np


def head_tail_ids(ids, max_len, sep_id, head_frac=0.5):
    # Keep the first part and the last part of a long document.
    # ids = 1D token ids without special tokens. Output length <= max_len, ends with sep_id.
    ids = np.asarray(ids)
    body_len = max_len - 1
    n = len(ids)
    if n <= body_len:
        out = ids.tolist()
        out.append(int(sep_id))
        return out
    head_n = int(body_len * head_frac)
    tail_n = body_len - head_n
    out = []
    if head_n > 0:
        out.extend(ids[:head_n].tolist())
    if tail_n > 0:
        out.extend(ids[n - tail_n:].tolist())
    out.append(int(sep_id))
    return out


def chunk_starts(n_tokens, chunk_len, overlap):
    # Start offsets of overlapping chunks that cover the whole document.
    body_len = chunk_len - 1
    stride = body_len - overlap
    if stride < 1:
        raise ValueError("overlap must be smaller than chunk_len - 1")
    starts = []
    start = 0
    while True:
        starts.append(start)
        if start + body_len >= n_tokens:
            break
        start = start + stride
    return starts


def pick_starts(starts, max_chunks, pick):
    # Reduce too many chunks to max_chunks.
    # pick: first | last | head_tail | uniform
    total = len(starts)
    if max_chunks is None or total <= max_chunks:
        return starts
    if pick == "first":
        return starts[:max_chunks]
    if pick == "last":
        return starts[total - max_chunks:]
    if pick == "head_tail":
        n_head = max_chunks // 2
        n_tail = max_chunks - n_head
        return starts[:n_head] + starts[total - n_tail:]
    if pick == "uniform":
        positions = np.linspace(0, total - 1, max_chunks)
        chosen = []
        for p in positions:
            idx = int(round(float(p)))
            if idx not in chosen:
                chosen.append(idx)
        result = []
        for idx in chosen:
            result.append(starts[idx])
        return result
    raise ValueError("unknown pick: " + str(pick))


def make_chunks(ids, sep_id, chunk_len, overlap, max_chunks=None, pick="uniform"):
    # Returns (chunks, rel_pos). Each chunk ends with sep_id.
    # rel_pos = where the chunk starts inside the document, 0..1 (use as a position feature).
    ids = np.asarray(ids)
    n = len(ids)
    body_len = chunk_len - 1
    starts = chunk_starts(n, chunk_len, overlap)
    starts = pick_starts(starts, max_chunks, pick)
    chunks = []
    rel_pos = []
    for s in starts:
        piece = ids[s:s + body_len].tolist()
        piece.append(int(sep_id))
        chunks.append(piece)
        rel_pos.append(s / max(1, n))
    return chunks, rel_pos


def pad_right(seqs, pad_id):
    # Right-pad a list of id lists. Returns input_ids [B, T], attention_mask [B, T], lengths [B].
    lengths = []
    for s in seqs:
        lengths.append(len(s))
    max_len = max(lengths)
    batch = len(seqs)
    input_ids = np.full((batch, max_len), int(pad_id), dtype=np.int64)
    mask = np.zeros((batch, max_len), dtype=np.int64)
    for i in range(batch):
        L = lengths[i]
        input_ids[i, :L] = np.asarray(seqs[i], dtype=np.int64)
        mask[i, :L] = 1
    return input_ids, mask, np.asarray(lengths, dtype=np.int64)


def multi_hot(label_lists, statute_ids):
    # label_lists: list of lists of statute IDs. statute_ids: ordered list of all statute IDs.
    # Returns a [N, S] int array, 1 where the statute is relevant.
    index_of = {}
    for j in range(len(statute_ids)):
        index_of[int(statute_ids[j])] = j
    Y = np.zeros((len(label_lists), len(statute_ids)), dtype=np.int64)
    for i in range(len(label_lists)):
        for s in label_lists[i]:
            if int(s) in index_of:
                Y[i, index_of[int(s)]] = 1
    return Y


def clean_prediction_ids(raw_ids, allowed_ids):
    # Keep only valid statute IDs, as unique integers. Order is kept (not a ranking).
    allowed = set(int(a) for a in allowed_ids)
    seen = set()
    out = []
    for x in raw_ids:
        v = int(x)
        if v in allowed and v not in seen:
            seen.add(v)
            out.append(v)
    return out


def macro_micro_f1(Y_true, Y_pred):
    # Y_true, Y_pred: [N, S] 0/1 arrays. Returns (macro_f1, micro_f1, subset_accuracy).
    Y_true = np.asarray(Y_true)
    Y_pred = np.asarray(Y_pred)
    f1_list = []
    tp_all = 0
    fp_all = 0
    fn_all = 0
    for s in range(Y_true.shape[1]):
        tp = int(((Y_true[:, s] == 1) & (Y_pred[:, s] == 1)).sum())
        fp = int(((Y_true[:, s] == 0) & (Y_pred[:, s] == 1)).sum())
        fn = int(((Y_true[:, s] == 1) & (Y_pred[:, s] == 0)).sum())
        tp_all += tp
        fp_all += fp
        fn_all += fn
        if tp == 0:
            f1_list.append(0.0)
        else:
            precision = tp / (tp + fp)
            recall = tp / (tp + fn)
            f1_list.append(2 * precision * recall / (precision + recall))
    macro = float(sum(f1_list) / max(1, len(f1_list)))
    if tp_all == 0:
        micro = 0.0
    else:
        p = tp_all / (tp_all + fp_all)
        r = tp_all / (tp_all + fn_all)
        micro = float(2 * p * r / (p + r))
    subset_acc = float((Y_true == Y_pred).all(axis=1).mean())
    return macro, micro, subset_acc


def best_global_threshold(probs, Y_true):
    # Scan one threshold for all statutes. Returns (best_threshold, macro_f1).
    probs = np.asarray(probs)
    best_t = 0.5
    best_f = -1.0
    for t in np.arange(0.05, 0.951, 0.01):
        pred = (probs >= t).astype(int)
        macro, _, _ = macro_micro_f1(Y_true, pred)
        if macro > best_f:
            best_f = macro
            best_t = float(t)
    return best_t, best_f


def write_submission(doc_ids, pred_lists, allowed_ids, path):
    # Codabench format: JSON list of {"id": ..., "pred": [unique int statute IDs]}, one per test doc.
    if len(doc_ids) != len(pred_lists):
        raise ValueError("doc_ids and pred_lists differ in length")
    rows = []
    for i in range(len(doc_ids)):
        clean_pred = clean_prediction_ids(pred_lists[i], allowed_ids)
        rows.append({"id": doc_ids[i], "pred": clean_pred})
    with open(path, "w", encoding="utf-8") as f:
        json.dump(rows, f)
    print("wrote", len(rows), "predictions to", path)

### 13b. Self-check of the helpers on real documents

In [ ]:
sys.path.insert(0, os.getcwd())
import importlib
import lsi_utils
importlib.reload(lsi_utils)
shutil.copy("lsi_utils.py", os.path.join(OUT_DIR, "lsi_utils.py"))

checked = 0
for split in ["train", "test"]:
    n_check = min(40, len(tokenized[split]))
    for i in range(n_check):
        ids = tokenized[split][i]
        n = len(ids)

        ht = lsi_utils.head_tail_ids(ids, HEAD_TAIL_LEN, SEP_ID, HEAD_FRAC)
        assert len(ht) <= HEAD_TAIL_LEN
        assert ht[-1] == SEP_ID
        if n <= HEAD_TAIL_LEN - 1:
            assert ht[:-1] == ids.tolist()

        chunks, rel = lsi_utils.make_chunks(ids, SEP_ID, CHUNK_LEN, CHUNK_OVERLAP, None, "uniform")
        covered = np.zeros(max(n, 1), dtype=bool)
        stride = CHUNK_LEN - 1 - CHUNK_OVERLAP
        for k in range(len(chunks)):
            assert chunks[k][-1] == SEP_ID
            assert len(chunks[k]) <= CHUNK_LEN
            start = k * stride
            body = chunks[k][:-1]
            assert body == ids[start:start + len(body)].tolist()
            covered[start:start + len(body)] = True
        if n > 0:
            assert covered.all()

        capped, rel2 = lsi_utils.make_chunks(ids, SEP_ID, CHUNK_LEN, CHUNK_OVERLAP, MAX_CHUNKS, "head_tail")
        assert len(capped) <= MAX_CHUNKS
        checked += 1
print("view self-check passed on", checked, "docs")

batch_ids, batch_mask, batch_len = lsi_utils.pad_right([[1, 2, 3], [4, 5]], PAD_ID)
assert batch_ids.shape == (2, 3) and batch_mask.sum() == 5
print("pad_right ok")

Y_demo = lsi_utils.multi_hot([[3, 7], [7]], [3, 7, 9])
assert Y_demo.tolist() == [[1, 1, 0], [0, 1, 0]]
assert lsi_utils.clean_prediction_ids([3, 3, 99, 7], [3, 7, 9]) == [3, 7]
print("multi_hot and clean_prediction_ids ok")

## 14. Save

In [ ]:
for split in SPLITS:
    folder = "Train" if split in ["train", "dev"] else "Test"
    folder_path = os.path.join(OUT_DIR, folder)
    os.makedirs(folder_path, exist_ok=True)
    rows_out = []
    for d in clean_docs[split]:
        row = {"id": d["id"], "text": d["text"]}
        if split != "test":
            row["labels"] = d["labels"]
        rows_out.append(row)
    with open(os.path.join(folder_path, split + ".json"), "w", encoding="utf-8") as f:
        json.dump(rows_out, f, ensure_ascii=False)

statute_universe = sorted(int(s) for s in pool_counts.keys())

bundle = {
    "vocab_size": VOCAB_SIZE,
    "sep_id": SEP_ID,
    "pad_id": PAD_ID,
    "id_dtype": ID_DTYPE.__name__,
    "tokenizer_dir": tok_dir,
    "statute_ids": statute_universe,
    "pos_weight": pos_weight,
    "config": {
        "seed": SEED, "n_folds": N_FOLDS, "chunk_len": CHUNK_LEN, "chunk_overlap": CHUNK_OVERLAP,
        "max_chunks": MAX_CHUNKS, "head_tail_len": HEAD_TAIL_LEN, "head_frac": HEAD_FRAC,
        "keep_newlines": KEEP_NEWLINES, "rare_threshold": RARE_THRESHOLD,
        "exclude_test_overlap_from_pool": EXCLUDE_TEST_OVERLAP_FROM_POOL,
    },
    "splits": {},
}
for split in SPLITS:
    labels = None
    if split != "test":
        labels = [d["labels"] for d in clean_docs[split]]
    bundle["splits"][split] = {
        "doc_ids": [d["id"] for d in clean_docs[split]],
        "ids": tokenized[split],
        "labels": labels,
        "n_chars": np.array([len(d["text"]) for d in clean_docs[split]], dtype=np.int64),
        "fold": fold_arrays[split],
    }

with open(os.path.join(OUT_DIR, "tokenized_lsi.pkl"), "wb") as f:
    pickle.dump(bundle, f, protocol=4)

with open(os.path.join(OUT_DIR, "audit_report.json"), "w") as f:
    json.dump(audit, f, indent=1)
label_table.to_csv(os.path.join(OUT_DIR, "statute_stats.csv"), index=False)
baseline_df.to_csv(os.path.join(OUT_DIR, "slice_baselines.csv"), index=False)
df_tok.to_csv(os.path.join(OUT_DIR, "doc_lengths.csv"), index=False)

meta = {
    "vocab_size": VOCAB_SIZE, "sep_id": SEP_ID, "pad_id": PAD_ID,
    "counts": {s: len(clean_docs[s]) for s in SPLITS},
    "n_statutes": len(statute_universe),
    "total_tokens": int(n_tokens_total),
}
with open(os.path.join(OUT_DIR, "meta.json"), "w") as f:
    json.dump(meta, f, indent=1)

for name in sorted(os.listdir(OUT_DIR)):
    full = os.path.join(OUT_DIR, name)
    if os.path.isfile(full):
        print(name.ljust(26), round(os.path.getsize(full) / 1e6, 2), "MB")

## 15. Verify the saved files

In [ ]:
with open(os.path.join(OUT_DIR, "tokenized_lsi.pkl"), "rb") as f:
    check = pickle.load(f)

for split in SPLITS:
    block = check["splits"][split]
    n = len(block["doc_ids"])
    assert len(block["ids"]) == n
    with open(os.path.join(OUT_DIR, "Train" if split != "test" else "Test", split + ".json"), "r", encoding="utf-8") as f:
        saved_rows = json.load(f)
    assert len(saved_rows) == n
    if split != "test":
        assert len(block["labels"]) == n
        for labs in block["labels"]:
            assert len(labs) > 0
            assert all(int(s) > 0 for s in labs)
            assert len(set(labs)) == len(labs)
    rng2 = np.random.RandomState(0)
    n_pick = min(15, n)
    if n_pick > 0:
        picks = rng2.choice(n, n_pick, replace=False)
        for i in picks:
            i = int(i)
            again = tokenizer(saved_rows[i]["text"], add_special_tokens=False)["input_ids"]
            assert again == block["ids"][i].tolist()
            assert str(saved_rows[i]["id"]) == str(block["doc_ids"][i])
    print(split, "OK |", n, "docs")

print("\nTest ids (first 3):", check["splits"]["test"]["doc_ids"][:3])
print("Statute universe size:", len(check["statute_ids"]))
print("\nALL CHECKS PASSED")

## 16. How later notebooks load this

```python
import pickle, sys
DATA = "/kaggle/input/<your-saved-output-dataset>"   # or /kaggle/working/lsi_data_clean
sys.path.insert(0, DATA)
import lsi_utils
with open(DATA + "/tokenized_lsi.pkl", "rb") as f:
    data = pickle.load(f)
SEP_ID, PAD_ID = data["sep_id"], data["pad_id"]
STATUTES = data["statute_ids"]                 # column order for the multi-hot targets
tr = data["splits"]["train"]                   # doc_ids, ids (np arrays), labels (lists), fold
Y = lsi_utils.multi_hot(tr["labels"], STATUTES)
chunks, rel_pos = lsi_utils.make_chunks(tr["ids"][0].astype("int64"), SEP_ID, 1024, 128, 16, "uniform")
```
Folds: `fold` is 0..4 for train and dev rows (train+dev pooled, stratified by rarest statute); -1 for test.
Submission: `lsi_utils.write_submission(test_doc_ids, pred_lists, STATUTES, "submission.json")`.